# EDA: IRA & CHIPS Act and US industrial investment

Question: did the two August 2022 laws make US industrial investment take off?
Unless stated otherwise, charts show one point per year. The dotted line marks 2022, the treatment year, and the last, hollow point is 2026, a year that is still incomplete.

In [1]:
import sys
from datetime import date

import polars as pl

sys.path.append('../src')
import data_processing as dp
from plot_functions import set_theme, plot_years, plot_treated_vs_controls, plot_small_multiples, plot_spaghetti,plot_monthly, plot_ranking

set_theme()

## 1. Data

In [2]:
data = dp.load_processed()
dp.overview(data)

file,rows,columns,start,end,% null
str,i64,i64,str,str,f64
"""census_construction_real_month…",200,18,"""2010-01-01""","""2026-08-01""",1.3
"""cim_state_investment_share_gdp""",51,4,"""-""","""-""",0.0
"""cim_totals_quarterly""",34,3,"""2018-01-01""","""2026-04-01""",0.0
"""cim_us_clean_investment_quarte…",34,42,"""2018-01-01""","""2026-04-01""",24.1
"""did_panel_monthly""",1800,6,"""2010-01-01""","""2026-08-01""",0.0
"""fred_monthly""",201,19,"""2010-01-01""","""2026-09-01""",0.2
"""oecd_donor_fit""",20,4,"""-""","""-""",0.0
"""oecd_gfcf_index2019_quarterly""",66,22,"""2010-01-01""","""2026-04-01""",0.0


The files read here are produced from `data/raw/` by [`src/data_processing.py`](../src/data_processing.py), which writes them to `data/processed/` (`uv run python src/data_processing.py`).
Census and FRED amounts are deflated by the construction materials price index (`WPUSI012011`), which soared in 2021-2022, and expressed in 2019 dollars.
The treated sector is computer, electronic and electrical factory construction (`mfg_computer_electronic_electrical`). The other manufacturing subsectors serve as control sectors, and the DiD panel adds offices and commercial.
For the OECD, each country's investment is rebased to 100 = 2019, and the 20 countries other than the United States form the synthetic control donor pool.

In [3]:
fred = data['fred_monthly']
census = data['census_construction_real_monthly']
oecd = data['oecd_gfcf_index2019_quarterly']
cim = data['cim_totals_quarterly']
MFG = dp.mfg_sectors(census)
TREATED = dp.TREATED_SECTOR
CONTROLS = [c for c in MFG if c != TREATED]

## 2. Factory construction

In [4]:
plot_years(fred, 'date', ['TLMFGCONS', 'TLMFGCONS_real'], scale=1000,
           title="US factory construction spending",
           subtitle='Annual average, $bn at an annual rate', ytitle='$bn')

In constant dollars, spending rises from **\$64bn in 2021 to \$175bn in 2024** (×2.7), then falls in 2025 and 2026.

In [5]:
plot_small_multiples(census, 'date', MFG, highlight=TREATED,
                     title='Construction by manufacturing subsector',
                     subtitle='Annual average, 2019 $M; each chart has its own scale')

The boom comes almost entirely from computer, electronic and electrical equipment (semiconductors, batteries). The other subsectors remain stable.

## 3. Treated sector and control sectors

In [6]:
plot_treated_vs_controls(census, 'date', TREATED, CONTROLS,
                         title='Treated sector and control sectors',
                         subtitle='Real construction, 100 = 2019')

Both groups move similarly in 2019-2021. By 2024, the treated sector is up tenfold while the control sectors gain about 45%.
Before 2019, the treated series fluctuates strongly, which weakens the parallel trends assumption over a long window.

In [7]:
plot_monthly(census.with_columns(pl.mean_horizontal(CONTROLS).alias('controls')), 'date', [TREATED, 'controls'],
             date(2020, 1, 1), date(2023, 12, 1), highlight=TREATED,
             title='Treated sector and control sectors, 2020-2023',
             subtitle='Monthly, 2019 $M at an annual rate', ytitle='2019 $M')

In monthly data, the rise starts in **late 2021**, before the laws. Treated-sector construction goes from \$8,020M in June 2021 to \$22,649M in June 2022 (×2.8), with a jump in October 2021 (from \$8,369M to \$12,967M), while the control-sector average stays flat (from \$6,653M to \$7,016M).
Two explanations are possible. The first is anticipation, since the CHIPS Act was debated in Congress from 2021 and Intel and TSMC announced their plants before the vote. The second is a sector-specific shock, the 2021 chip shortage.
Either way, the parallel trends assumption with a treatment dated August 2022 is threatened.

## 4. Cross-country comparison

In [8]:
plot_spaghetti(oecd, 'quarter', 'USA',
               title="Investment in non-residential structures, United States and 20 OECD countries",
               subtitle='Volume, 100 = 2019')

At the country level, the United States stays within the bundle of other countries. The effect is diluted in an aggregate that includes offices and infrastructure.

In [9]:
data['oecd_donor_fit']

country,corr_pre,rmse_pre,index_post
str,f64,f64,f64
"""IRL""",0.67,10.06,78.07
"""ESP""",0.55,4.96,112.67
"""DEU""",0.49,5.6,93.01
"""FIN""",0.46,6.56,91.1
"""SWE""",0.45,8.81,102.68
…,…,…,…
"""AUS""",-0.04,15.73,112.83
"""ISR""",-0.04,15.23,116.59
"""NOR""",-0.08,9.82,117.77


From 2015-Q1 to 2022-Q2, Spain (correlation 0.55, RMSE 4.96), Germany (0.49 and 5.6), Finland (0.46) and Sweden (0.45) track the United States best. Ireland has the highest correlation (0.67) but a high RMSE (10.06), a sign of a very volatile series.
Some countries move against the United States, in particular Denmark (−0.52) and Italy (−0.31, average index of 183 since 2024), whose trajectory clearly diverges. The synthetic control nevertheless keeps the full pool of 20 countries and lets the weights select the donors, which gives Denmark and Italy a zero weight; restricting the pool to the best-correlated countries is tested as a robustness check (`synthetic_control.ipynb`).

## 5. Clean tech factories

In [10]:
plot_years(cim, 'quarter', ['announced', 'actual'], how='sum',
           title='Clean tech factories, announced and actual investment',
           subtitle='Annual total, 2024 $bn (complete years)', ytitle='$bn')

Announcements peak in 2022 at **\$91bn**, the year of the IRA. Actual spending follows with a one- to two-year lag and peaks at \$49bn in 2024.

In [11]:
plot_ranking(data['cim_state_investment_share_gdp'], 'state', 'pct_gdp_manufacturing', n=15,
             title='Clean tech factories, 15 most exposed states',
             subtitle="Clean tech manufacturing investment over the last 4 quarters, % of state GDP",
             xtitle='% of GDP')

Investment is concentrated in a few states. Nevada (0.99% of GDP) and Kentucky (0.77%) lead South Carolina, Georgia and Tennessee, followed by the Midwest (Indiana, Kansas, Michigan). This share could serve as a state-level measure of treatment exposure in a continuous-treatment DiD.

## 6. Confounders

In [12]:
plot_small_multiples(fred, 'date', ['TLMFGCONS_real', 'FEDFUNDS', 'GS10', 'WPUSI012011', 'IPG3344S', 'CES3133440001'],
                     title='Construction, rates, prices and downstream effects',
                     subtitle='Annual average; each chart has its own scale')

The boom happens during the rate hikes, which if anything biases the estimated effect downwards.
Semiconductor output rises, but employment in the sector barely moves.

## 7. Takeaways

- **Concentrated boom**: factory construction rises ×2.7 between 2021 and 2024, driven almost entirely by the computer, electronic and electrical sector (×10 against +45% for the other subsectors). This contrast is the basis of the DiD in `did.ipynb`, which compares the treated sector with the 6 other manufacturing subsectors, in logs.
- **Rise before the laws**: treated-sector spending starts climbing in late 2021, through anticipation or the chip shortage. `did.ipynb` therefore uses the mid-2019 to mid-2021 plateau as its before period, excludes the transition months, checks the dynamics with an event study and reports a range rather than a single effect.
- **Volatile long pre-period**: before 2019 the treated series swings strongly, so a long before window is fragile; `did.ipynb` shows its result as a robustness check.
- **Diluted country-level signal**: the United States stays within the bundle of OECD countries on a broad non-residential aggregate, so `synthetic_control.ipynb` should find a much smaller effect than the sector-level DiD.
- **Donors**: `synthetic_control.ipynb` uses the full pool of 20 countries and lets the weights select the donors; restricting the pool to the best-correlated countries (`oecd_donor_fit` table) is tested as a robustness check.
- **Confounders**: materials prices are handled by deflation, and shocks common to all sectors (rates, prices) are removed by the DiD comparison across sectors. The rate hike coincides with the treatment and, if anything, biases the effect downwards.